https://cobrapy.readthedocs.io/en/latest/solvers.html

13. Solvers

In [13]:
from cobra.io import load_model
model = load_model('textbook')

model.solver = 'glpk'
# or if you have cplex installed
model.solver = 'cplex'

SolverNotFound: cplex is not a valid solver interface. Pick one from glpk_exact, glpk, scipy.

In [14]:
type(model.solver)

optlang.glpk_interface.Model

https://portal.gurobi.com/iam/licenses/request/
http://www.gurobi.cn/NewsView1.Asp?id=4

https://cn.bing.com/search?q=cplex&qs=n&form=QBRE&sp=-1&lq=0&pq=cplex&sc=12-5&sk=&cvid=16CB2D7F3B8645BC905EF71EA48F7C22

https://cobrapy.readthedocs.io/en/latest/constraints_objectives.html

14. Tailored constraints, variables and objectives

In [1]:
from cobra.io import load_model

model = load_model('textbook')

In [2]:
model

Name,e_coli_core
Memory address,7f80cb3b12b0
Number of metabolites,72
Number of reactions,95
Number of genes,137
Number of groups,0
Objective expression,1.0*Biomass_Ecoli_core - 1.0*Biomass_Ecoli_core_reverse_2cdba
Compartments,"cytosol, extracellular"


In [3]:
model.reactions

[<Reaction ACALD at 0x7f80cb32f760>,
 <Reaction ACALDt at 0x7f80cb32f730>,
 <Reaction ACKr at 0x7f80cb32f250>,
 <Reaction ACONTa at 0x7f80cb32fb20>,
 <Reaction ACONTb at 0x7f80cb33ca90>,
 <Reaction ACt2r at 0x7f80cb33c7f0>,
 <Reaction ADK1 at 0x7f80cb32ffa0>,
 <Reaction AKGDH at 0x7f80cb32feb0>,
 <Reaction AKGt2r at 0x7f80cb2c7760>,
 <Reaction ALCD2x at 0x7f80cb2c7f10>,
 <Reaction ATPM at 0x7f80cb2c7670>,
 <Reaction ATPS4r at 0x7f80cb33c8b0>,
 <Reaction Biomass_Ecoli_core at 0x7f80cb2c7c70>,
 <Reaction CO2t at 0x7f80cb2c7970>,
 <Reaction CS at 0x7f80cb32fa00>,
 <Reaction CYTBD at 0x7f80cb2d90a0>,
 <Reaction D_LACt2 at 0x7f80cb2e0b50>,
 <Reaction ENO at 0x7f80cb2e0cd0>,
 <Reaction ETOHt2r at 0x7f80cb32fa60>,
 <Reaction EX_ac_e at 0x7f80cb2e0d30>,
 <Reaction EX_acald_e at 0x7f80cb2d9250>,
 <Reaction EX_akg_e at 0x7f80cb2cef10>,
 <Reaction EX_co2_e at 0x7f80cb2ceee0>,
 <Reaction EX_etoh_e at 0x7f80cb2eaf70>,
 <Reaction EX_for_e at 0x7f80cb2eafa0>,
 <Reaction EX_fru_e at 0x7f80cb2eaee0>,
 

In [4]:
type(model.reactions)

cobra.core.dictlist.DictList

In [5]:
model.reactions.FBA

Reaction identifier,FBA
Name,fructose-bisphosphate aldolase
Memory address,0x7f80cb2f21f0
Stoichiometry,"fdp_c <=> dhap_c + g3p_c D-Fructose 1,6-bisphosphate <=> Dihydroxyacetone phosphate + Glyceraldehyde 3-phosphate"
GPR,b1773 or b2097 or b2925
Lower bound,-1000.0
Upper bound,1000.0


https://cobrapy.readthedocs.io/en/latest/autoapi/cobra/core/reaction/index.html

property flux_expression: Optional[optlang.interface.Variable][source]
    Get Forward flux expression.

    Returns: flux_expression – The expression representing the the forward flux (if associated with model), otherwise None. Representing the net flux if model.reversible_encoding == ‘unsplit’ or None if reaction is not associated with a model

    Return type: optlang.interface.Variable, optional

In [6]:
model.reactions.FBA.flux_expression

1.0*FBA - 1.0*FBA_reverse_84806

In [7]:
same_flux = model.problem.Constraint(
    model.reactions.FBA.flux_expression - model.reactions.NH4t.flux_expression,
    lb=0,
    ub=0)
model.add_cons_vars(same_flux)

In [8]:
solution = model.optimize()
print(solution.fluxes['FBA'], solution.fluxes['NH4t'],
      solution.objective_value)

4.662749047738146 4.662749047738147 0.8551109609261567


---

In [10]:
coefficients = dict()
for rxn in model.reactions:
    coefficients[rxn.forward_variable] = 1.
    coefficients[rxn.reverse_variable] = 1.
constraint = model.problem.Constraint(0, lb=0, ub=100)
model.add_cons_vars(constraint)
model.solver.update()
constraint.set_linear_coefficients(coefficients=coefficients)

In [ ]:
# https://optlang.readthedocs.io/en/latest/_modules/optlang/interface.html#Objective.set_linear_coefficients

def set_linear_coefficients(self, coefficients):
    """Set coefficients of linear terms in constraint or objective.
    Existing coefficients for linear or non-linear terms will not be modified.

    Note: This method interacts with the low-level solver backend and can only be used on objects that are
    associated with a Model. The method is not part of optlangs basic interface and should be used mainly where
    speed is important.

    Parameters
    ----------
    coefficients : dict
        A dictionary like {variable1: coefficient1, variable2: coefficient2, ...}

    Returns
    -------
    None
    """
    raise NotImplementedError("Child classes should implement this.")

### **Q1：解释一下这段代码**

这段代码展示了如何**批量添加复杂约束**，特别是添加一个约束条件：模型中所有反应通量的绝对值之和不超过 100。具体来说，它通过以下步骤实现：

#### **1. 数学背景**
- **目标约束**：  
  $$
  \sum_{\text{所有反应}} |v_i| \leq 100
  $$
  其中 $v_i$ 是反应 $i$ 的净通量（正向通量 - 反向通量）。由于绝对值的非线性特性，直接建模困难，因此代码采用了一种线性化技巧。

#### **2. 线性化技巧**
在 cobrapy 中，每个反应的通量 $v_i$ 被分解为正向变量（`forward_variable`）和反向变量（`reverse_variable`），且两者均为非负数。例如：
$$
v_i = \text{forward\_variable} - \text{reverse\_variable}
$$
此时，通量的绝对值可近似为：
$$
|v_i| \approx \text{forward\_variable} + \text{reverse\_variable}
$$
因此，总绝对值之和的约束变为：
$$
\sum_{\text{所有反应}} (\text{forward\_variable} + \text{reverse\_variable}) \leq 100
$$

#### **3. 代码步骤**
- **步骤 1：构建系数字典**  
  为每个反应的正向和反向变量赋予系数 1：
  ```python
  coefficients = dict()
  for rxn in model.reactions:
      coefficients[rxn.forward_variable] = 1.0  # 正向变量系数为 1
      coefficients[rxn.reverse_variable] = 1.0  # 反向变量系数为 1
  ```
  这相当于将每个反应的 `forward_variable` 和 `reverse_variable` 的贡献相加。

- **步骤 2：创建空约束**  
  初始化一个上下界为 [0, 100] 的约束，但尚未关联变量：
  ```python
  constraint = model.problem.Constraint(0, lb=0, ub=100)  # 初始表达式为 0
  model.add_cons_vars(constraint)  # 将约束添加到模型
  ```

- **步骤 3：更新求解器**  
  调用 `model.solver.update()`，确保求解器识别新添加的约束：
  ```python
  model.solver.update()
  ```

- **步骤 4：设置线性系数**  
  将之前构建的系数字典应用到约束中：
  ```python
  constraint.set_linear_coefficients(coefficients=coefficients)
  ```
  此时，约束的实际表达式变为：
  $$
  \sum (\text{forward\_variable} + \text{reverse\_variable}) \leq 100
  $$

#### **4. 意义**
- **高效性**：对于大型模型，逐个添加约束效率低下。通过一次性设置所有系数，减少与求解器的交互次数。
- **避免重复代码**：适用于需要为大量变量添加相同类型约束的场景（如全局资源限制）。

---

### **Q2：`model.problem` 返回的是什么？**

`model.problem` 是 cobrapy 模型的底层数学优化接口，**返回一个 `optlang` 的 `Model` 对象**。  
- **optlang**：一个抽象层，支持多种求解器（如 GLPK、CPLEX、Gurobi），提供统一的 API。
- **作用**：  
  通过 `model.problem`，可以直接操作底层优化模型，例如：
  - 添加变量 (`model.problem.Variable`)
  - 添加约束 (`model.problem.Constraint`)
  - 修改目标函数 (`model.problem.Objective`)

#### **示例**
```python
# 添加一个变量
var = model.problem.Variable("my_var", lb=0)
model.add_cons_vars(var)

# 添加一个约束
constraint = model.problem.Constraint(var, lb=0, ub=10)
model.add_cons_vars(constraint)
```

---

### **Q3：`model.solver.update()` 是什么意思？为什么有这行代码？**

`model.solver.update()` 的作用是 **通知求解器同步模型的最新状态**。具体来说：

#### **1. 为什么需要同步？**
- 当通过 cobrapy 或 optlang 修改模型（如添加变量、约束）时，这些更改不会立即传递给底层的求解器（如 GLPK）。
- 求解器内部维护自己的模型副本，需要显式触发更新以反映最新修改。

#### **2. 何时需要调用？**
- **修改模型结构后**：例如添加/删除变量、约束。
- **设置系数前**：在调用 `set_linear_coefficients()` 前，确保约束已被求解器识别。

#### **3. 示例流程**
```python
# 添加约束到 cobrapy 模型
model.add_cons_vars(constraint)

# 同步到求解器
model.solver.update()

# 设置约束的系数（此时约束已在求解器中存在）
constraint.set_linear_coefficients(coefficients)
```

#### **4. 不调用的后果**
- 可能引发错误：例如尝试为不存在的约束设置系数。
- 求解结果不准确：求解器基于旧模型计算。

---

### **总结**
| 问题 | 核心解释 |
|------|----------|
| **Q1** | 通过批量设置系数，高效添加全局约束（总通量绝对值之和 ≤ 100）。 |
| **Q2** | `model.problem` 返回底层 optlang 的 Model 对象，用于直接操作优化问题。 |
| **Q3** | `model.solver.update()` 是同步模型到求解器的关键步骤，避免操作未识别的变量或约束。 |

实际应用中，这种方法是构建复杂约束（如酶约束、代谢流量平衡）的标准操作，尤其适用于基因组规模代谢模型（GEMs）的扩展。

In [18]:
model = load_model('textbook')
difference = model.problem.Variable('difference')

In [19]:
constraint = model.problem.Constraint(
    model.reactions.EX_glc__D_e.flux_expression -
    model.reactions.EX_nh4_e.flux_expression - difference,
    lb=0,
    ub=0)
model.add_cons_vars([difference, constraint])

In [20]:
for reaction in model.reactions[:5]:
    with model:
        reaction.knock_out()
        model.optimize()
        print(model.solver.variables.difference.primal)

-5.234680806802557
-5.234680806802542
-5.234680806802542
-10.000000000000028
-10.0
